# GES quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/ges_quickstart.ipynb)

Greedy Equivalence Search (GES) recovers the CPDAG of the Markov-equivalence class from
observational linear-Gaussian data. We generate a three-node chain and recover its structure.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.6.0" pandas

In [2]:
import numpy as np
import pandas as pd

import andrey

A linear-Gaussian structural equation model -- a chain `x0 -> x1 -> x2`. The columns come in an
order drawn from the seed, so their order says nothing about the causal order; `shuffle=False`
keeps `x0, x1, x2`.

In [3]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges, cause first


def sample(seed: int, n: int = 1500, shuffle: bool = True) -> pd.DataFrame:
    """Draw n rows of the chain, its columns in an order drawn from the seed."""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = 0.9 * x0 + rng.standard_normal(n)
    x2 = 0.9 * x1 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2})
    return frame[rng.permutation(frame.columns)] if shuffle else frame


data = sample(seed=0)
print("columns:", list(data.columns))

columns: ['x0', 'x2', 'x1']


Fit GES and inspect the recovered CPDAG (endpoint-mark adjacency) and its BIC score. The chain is
Markov-equivalent to its undirected skeleton, so the CPDAG leaves the two edges unoriented.

In [4]:
out = andrey.ges(data)
print("kind:", out.structure.kind)
print("CPDAG adjacency, in column order:")
print(out.structure.to_numpy())
print("edges:", out.structure.oriented_edges())
print("BIC score:", round(out.metadata["score"], 4))

kind: cpdag
CPDAG adjacency, in column order:
[[0 0 1]
 [0 0 1]
 [1 1 0]]
edges: [('x0', 'x1', 'undirected'), ('x2', 'x1', 'undirected')]
BIC score: 1.0951


In [5]:
import andrey.viz

andrey.viz.draw(out.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="142.25" y1="88.86" x2="217.75" y2="115.65" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="142.25" y1="151.14" x2="217.75" y2="124.35" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><circle cx="130.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text><circle cx="130.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text><circle cx="230.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text></svg>'